In [2]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "code":
    chdir("..")
print(f"CWD: {Path.cwd()}")

CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main/code
CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [3]:
from src.models.MatrixFactorization import MF, UMF
from src.graphs import random_k_out_graph, create_graph
from src.users import User
from src.training.decentralized import decentralized_train_loop, decentralized_validate_loop, decentralized_train_n_epochs
from src.data_utils import create_batched_dataloaders, create_dataloader

In [4]:
#Make data sample iterable
from torch.utils.data import Dataset, DataLoader, TensorDataset, Sampler
import torch
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim import SGD

from collections import Counter
import networkx as nx
from networkx.generators.classic import empty_graph
from networkx.utils import discrete_sequence, py_random_state, weighted_choice

import seaborn as sns

## Parameter Setting

In [6]:
model = "umf"
val_loader_type = "rs"
train_loader_type = "rs"
userprop = None
n_factors = 30
sparse = False
batch_size = 10
lr = 0.03871364416669273
weight_decay = 0.14214480688557163
graph_seed = 1
n_epochs = 150

## Main

In [56]:
train_df = pd.read_csv("dataset/ml100k_train_seed10.csv")
test_df = pd.read_csv("dataset/ml100k_test_seed10.csv")
train_df.head()

,user_id,item_id,rating
0,871,974,3
1,337,487,5
2,278,383,4
3,641,868,3
4,428,434,4


In [58]:
n_users = train_df['user_id'].nunique()
n_items = train_df['item_id'].nunique()
print(f"Total User: {n_users}")
print(f"Total Item: {n_items}")

Total User: 943
Total Item: 1633


In [60]:
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=0)
train_data_loader = create_dataloader(
        df=train_df, dl_type=train_loader_type, batch_size=batch_size, p=userprop
    )
val_data_loader = create_dataloader(df=val_df, dl_type=val_loader_type)
test_data_loader = create_dataloader(df=test_df, dl_type=val_loader_type)


In [62]:
users = {}
for i in tqdm(range(n_users)):
    # model = MF(n_users=n_users, n_items=n_items)
    user_model = UMF(n_items, n_factors = n_factors, sparse = sparse)
    # model = GeneralizedMFOneLayer(n_users=n_users, n_items=n_items)
    users[i] = User(id=i, model=user_model, optimizer=SGD(user_model.parameters(), lr=lr, weight_decay=weight_decay), model_name = model)

  0%|          | 0/943 [00:00<?, ?it/s]

In [63]:
graph = random_k_out_graph(n=n_users, k=2, seed=graph_seed)

In [65]:
train_losses, val_losses, time_per_epoch, commutes = decentralized_train_n_epochs(
        user_models=users,
        train_loader=train_data_loader,
        val_loader=val_data_loader,
        epochs=n_epochs,
        graph=graph,
    )

  0%|          | 0/60000 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.6333 | Validation Loss: 1.3908 | Time Elapsed: 39.898963 sec |Commute: 119807 |

  0%|          | 0/60000 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 0.2644 | Validation Loss: 1.1628 | Time Elapsed: 41.148934 sec |Commute: 119807 |

  0%|          | 0/60000 [00:00<?, ?it/s]

...

Epoch 12 | Train Loss: 0.2789 | Validation Loss: 0.9129 | Time Elapsed: 39.002669 sec |Commute: 119807 |

  0%|          | 0/60000 [00:00<?, ?it/s]

Epoch 13 | Train Loss: 0.2793 | Validation Loss: 0.9247 | Time Elapsed: 38.459556 sec |Commute: 119807 |

Early stopping.

Total time elapsed: 509.8244883330044

In [72]:
test_loss = decentralized_validate_loop(users, test_data_loader)

In [73]:
test_loss

0.916328